# Week 3: Reference-label evaluation, mitigation, and calibration

You now receive:

- `week3_validation_reference.csv`
- `week3_test_observed.csv`
- `week3_test_reference.csv`

These files let you compare evaluation against the labels available to the modelling pipeline, $Y^{obs}$, and the best available reference labels, $Y^{ref}$.

**Rules**

- Do not train a predictive model on `income_reference`.
- Reference validation may be used for final model selection or calibration only if you state that design choice.
- Reference test is used once for final evaluation.
- Preserve `row_id` when joining files and verify one-to-one joins.

In [ ]:
%pip -q install fairlearn cleanlab scikit-learn pandas matplotlib seaborn

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RANDOM_STATE = 42
DATA_DIR = Path("/content/data")

train = pd.read_csv(DATA_DIR / "week2_train_observed.csv")
val_obs = pd.read_csv(DATA_DIR / "week2_validation_observed.csv")
val_ref = pd.read_csv(DATA_DIR / "week3_validation_reference.csv")
test_obs = pd.read_csv(DATA_DIR / "week3_test_observed.csv")
test_ref = pd.read_csv(DATA_DIR / "week3_test_reference.csv")

# TODO: assert unique row_id values and exact alignment of feature columns.
print(train.shape, val_obs.shape, val_ref.shape, test_obs.shape, test_ref.shape)

## 1. Quantify the label gap

Compare $Y^{obs}$ and $Y^{ref}$ on validation and test data. Report overall disagreement, disagreement conditional on each label, group-specific disagreement, and intersections. Inspect which feature regions contain disagreements without assuming the reference label is infallible.

In [ ]:
# TODO: join observed and reference files by row_id.
# TODO: quantify and visualize label disagreement.
# TODO: test the Week 2 hypotheses without reading instructor metadata.

## 2. Re-evaluate the original model twice

For the same scores and decisions, produce two complete audits:

1. against `income_observed`;
2. against `income_reference`.

Any difference is caused by the evaluation target, not by a changed model. Discuss which conclusions reverse or materially change.

In [ ]:
# TODO: reconstruct your Week 2 baseline.
# TODO: create a reusable audit function accepting y_eval, scores, predictions, and sensitive features.
# TODO: report utility, independence, separation, sufficiency, and calibration twice.

## 3. Design mitigation candidates

Compare at least five deliberately chosen candidates:

| Candidate | Purpose |
|---|---|
| M0: all-feature baseline | Reproduce the original pipeline |
| M1: remove sensitive attributes from prediction | Test direct-use ablation, not a fairness guarantee |
| M2: remove or neutralize suspected unstable feature(s) | Test shortcut dependence |
| M3: Cleanlab-informed filter or reweighting | Test label-quality mitigation |
| M4: combined data/feature mitigation | Test interaction between failures |
| M5: Fairlearn constraint or postprocessing | Test an explicitly chosen parity constraint |

You may add models. Every intervention requires a causal or operational rationale and a documented cost.

In [ ]:
# TODO: define the candidates and a common experiment protocol.
# Keep preprocessing, split, random seed, and evaluation consistent.

### Cleanlab pattern: out-of-fold auditing

Complete this template. Filtering is not always appropriate. Reweighting or manual review may be safer when errors are uncertain or group-dependent.

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from cleanlab.filter import find_label_issues

# cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
# oof_prob = cross_val_predict(..., method="predict_proba", cv=cv, n_jobs=-1)
# ranked = find_label_issues(labels=..., pred_probs=oof_prob,
#                            return_indices_ranked_by="self_confidence")

# TODO: predeclare how many examples to review/filter or how weights are constructed.
# TODO: compare the demographic composition of affected and unaffected rows.

### Fairlearn patterns

`ThresholdOptimizer` changes decisions, not probability calibration. `ExponentiatedGradient` trains a randomized classifier under a constraint. Equalized odds is evaluated relative to the label supplied during fitting.

In [ ]:
from fairlearn.postprocessing import ThresholdOptimizer
from fairlearn.reductions import EqualizedOdds, ExponentiatedGradient
from sklearn.linear_model import LogisticRegression

# TODO: choose which validation label is appropriate for fitting a postprocessor.
# post = ThresholdOptimizer(estimator=base_model, constraints="equalized_odds", predict_method="predict_proba")
# post.fit(X_val, y_val_..., sensitive_features=A_val)

# For ExponentiatedGradient, transform features first. Some Fairlearn/sklearn
# combinations require a dense array:
# fitted_preprocessor = base_model.named_steps["preprocess"]
# X_train_encoded = fitted_preprocessor.transform(X_train)
# if hasattr(X_train_encoded, "toarray"):
#     X_train_encoded = X_train_encoded.toarray()
# eg = ExponentiatedGradient(LogisticRegression(max_iter=1000), EqualizedOdds())
# eg.fit(X_train_encoded, y_train_observed, sensitive_features=A_train)

# TODO: implement, justify, and audit one Fairlearn intervention.

## 4. Calibration experiment

Compare at least these score pipelines:

1. uncalibrated model;
2. calibration fitted to observed validation labels;
3. calibration fitted to reference validation labels.

Evaluate each against both observed and reference test labels. Keep calibration separate from decision postprocessing. Explain why a score can be calibrated against one label and miscalibrated against another.

In [ ]:
from sklearn.calibration import CalibratedClassifierCV
from sklearn.frozen import FrozenEstimator

# Current sklearn pattern for a pre-fitted estimator. The calibration set
# must be disjoint from the model-fitting set:
# calibrated_obs = CalibratedClassifierCV(
#     estimator=FrozenEstimator(base_model), method="sigmoid"
# )
# calibrated_obs.fit(X_val, y_val_observed)

# TODO: implement all three pipelines, reliability plots, ECE, log loss, and Brier score.

## 5. Final locked test evaluation

Select the final candidates before examining reference test results. Then produce one comparison table with:

- performance against $Y^{obs}$ and $Y^{ref}$;
- sex-specific and intersectional counts;
- selection-rate gap;
- TPR and FPR gaps;
- PPV and NPV gaps;
- AUROC, log loss, Brier, and calibration error;
- bootstrap intervals for key gaps;
- number of training examples removed/reweighted;
- features required at deployment.

In [ ]:
# TODO: pre-register selected candidates in this list before loading/using reference test labels.
SELECTED_CANDIDATES = []

# TODO: run the locked evaluation and save the result table.

## Questions to think about!

1. Which Week 2 conclusions were robust to the reference-label audit?
2. Which apparent performance or fairness results were artifacts of $Y^{obs}$?
3. How did the suspected shortcut affect in-distribution validation and shifted test performance?
4. Did Cleanlab find injected problems, natural hard cases, both, or neither? How do you know?
5. Which mitigation improved reference performance, and what did it cost?
6. Did a parity intervention improve the chosen harm-related metric against the correct label?
7. How did observed-label and reference-label calibration differ?
8. Why can equalized odds and calibration conflict when group base rates differ?
9. What evidence would be required before changing real labels or deploying group-specific thresholds?
10. What remains unvalidated?

The report must distinguish model behaviour, label quality, data shift, fairness criteria, and normative judgement.

## Final Report for Project 1: Auditing & Mitigating Hidden Bias

1. Submit a 3-4 page report (strict; additional pages will lead to a warning and then 'Fail') using CVPR paper template - https://github.com/cvpr-org/author-kit/releases
2. Project title, group members and declaration of AI usage (if any)

## Recommended Strcuture 

1. The prediction task, dataset, sensitive attribute, and outcome.
2. The available training, validation, and test splits.
3. The distinction between:
    - (Y_{\mathrm{obs}}): the observed label available during model development.
    - (Y_{\mathrm{ref}}): the cleaner reference label used to estimate true performance.
4. Which labels were available during each week.

Use ~0.5 page

### Week 1: Baseline Fairness Audit (~1 page)

1. Summarize the guided analysis of the original dataset.
2. Overall and group specific predictive performance.
3. Fairness metrics considered, such as selection rate, TP/FP rates, etc.
4. Why different metrics describe different notion of fairness?
5. Wheather the model satisfies one criterion while violating another.
6. Report only revelant metrics and explain their application.

### Week 2: Blind audit of dataset using observed labels (~1 page)

1. Explain how the investigation do before receiving the acutal reference labels.
2. What are your initial hypotheses about possible issues in the dataset
    - Label-quality problems? group-dependent label errors? label bias? sho... le...?
3. What methods you used to audit? For each method state what it is intended to detect and its main assumptions?
4. Present your strongest evidence you found.
    - Distinguish between a suspicious association
    - Evidence consistent with label bias
    - Evidence consistent with shortcut learning
    - Any conclusion?
5. Can you trust Cleanlab scores? Why or why not?

### Week 3: Reference label evaluation and mitigation (~1.5 page)

1. Reveal the hidden data problem using Y_ref (gold-standard labels), quantify:
    - overall disagreement between observed and reference labels
    - disagreement rates by sensitive group (along with direction of disagreement)
2. Wheather your Week 2 audit successfully identified the biased (affect) rows, groups or features?
3. What difference you see in observed and reference performance
    - Evaluate the same model predictions against both labels
4. Discuss how conclusions about performance, fairness and calibration change when the evaluation label changes. State which result represents apparent obsereved performance and which is best estimate of reference performance.

5. Mitigation - 
    - Compare a small, justified set of intervention you followed (like removing suspected shortcut, sensitive feature, etc.)
    - For every intervention report - what data is it fit on, labels used, fairness criterion it targets, effect on performance, fairness and calibration and importantly, adverese effects.
6. Finally, model selection and final test
    - Explain how final candidate was selected using validation data. Declare selected model before presenting test results.

### Discussion and conclusion (~0.5 pages)

1. Required visual evidence; whenever necessary throughout the report -- should be report with table or figure with descriptive caption
2. Focus on interpretation rather than notebook code
3. Also report relevant negative or inconclusive findings
